<a href="https://colab.research.google.com/github/AlexandreStackholders/AnhangueraADS/blob/main/Projeto_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import datetime
import os
import sqlite3

# Nome do arquivo do banco de dados SQLite
DB_FILE = "clinica.db"


## ⚙️ FUNÇÕES DE ACESSO AO BANCO DE DADOS (DB)
def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    """Inicializa o banco de dados e cria a tabela 'pacientes' se ela não existir."""
    try:
        with get_db_connection() as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                CREATE TABLE IF NOT EXISTS pacientes (
                    id INTEGER PRIMARY KEY,
                    nome TEXT NOT NULL,
                    cpf TEXT UNIQUE NOT NULL,
                    idade INTEGER NOT NULL,
                    telefone TEXT,
                    endereco TEXT
                )
            """
            )
            conn.commit()
        print(f"Banco de dados '{DB_FILE}' verificado/inicializado.")
    except Exception as e:
        print(f"ERRO ao inicializar o banco: {e}")


def get_pacientes_para_exibir(termo_busca=None):
    with get_db_connection() as conn:
        cursor = conn.cursor()
        if termo_busca:
            cursor.execute(
                "SELECT * FROM pacientes WHERE nome LIKE ? ORDER BY nome",
                ("%" + termo_busca + "%",),
            )
        else:
            cursor.execute("SELECT * FROM pacientes ORDER BY nome")
        encontrados = cursor.fetchall()
    return [dict(p) for p in encontrados]


## 🖥️ FUNÇÕES DE APRESENTAÇÃO
def print_box(title, content_lines, force_width=78):
    box_width = force_width
    content_width = box_width - 4
    print("\n╔" + "═" * (box_width - 2) + "╗")
    print(f"║ {title.center(content_width)} ║")
    if content_lines:
        print("╠" + "═" * (box_width - 2) + "╣")
        for line in content_lines:
            print(f"║ {line[:content_width].ljust(content_width)} ║")
    print("╚" + "═" * (box_width - 2) + "╝")


def validar_int_positivo(prompt, valor_atual=None):
    while True:
        entrada = input(prompt)
        if not entrada and valor_atual is not None:
            return valor_atual
        if entrada.isdigit() and int(entrada) > 0:
            return int(entrada)
        print("Entrada inválida. Digite um número positivo.")


def selecionar_paciente(pacientes):
    if not pacientes:
        return None
    while True:
        escolha = input("\nDigite o NÚMERO do paciente: ")
        if escolha.isdigit() and 0 < int(escolha) <= len(pacientes):
            return pacientes[int(escolha) - 1]
        print("Número inválido.")


## ➕ FUNÇÕES DE MANIPULAÇÃO DE DADOS
def cadastrar_paciente():
    print("\n--- CADASTRAR PACIENTE ---")
    try:
        nome = input("Nome do paciente: ")
        cpf = input("CPF (somente números): ")
        idade = validar_int_positivo("Idade: ")
        telefone = input("Telefone: ")
        endereco = input("Endereço Completo: ")

        with get_db_connection() as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                INSERT INTO pacientes (nome, cpf, idade, telefone, endereco)
                VALUES (?, ?, ?, ?, ?)
            """,
                (nome, cpf, idade, telefone, endereco),
            )
            conn.commit()
        print("✅ Paciente cadastrado com sucesso!")
    except sqlite3.IntegrityError:
        print("❌ ERRO: CPF já cadastrado.")
    except Exception as e:
        print(f"❌ ERRO: {e}")


def buscar_paciente():
    print("\n--- BUSCAR PACIENTE ---")
    termo = input("Digite o nome para busca: ")
    encontrados = get_pacientes_para_exibir(termo)
    if encontrados:
        for p in encontrados:
            print(
                f"ID: {p['id']} | Nome: {p['nome']} | Endereço: {p['endereco']}"
            )
    else:
        print("Nenhum paciente encontrado.")


def listar_todos_pacientes():
    pacientes = get_pacientes_para_exibir()
    print("\n--- LISTA DE PACIENTES ---")
    if not pacientes:
        print("Nenhum paciente cadastrado.")
    else:
        for i, p in enumerate(pacientes):
            print(
                f"[{i+1}] Nome: {p['nome']} | CPF: {p['cpf']} | Endereço: {p['endereco']}"
            )
    return pacientes


def editar_paciente():
    print("\n--- EDITAR PACIENTE ---")
    pacientes = listar_todos_pacientes()
    p = selecionar_paciente(pacientes)
    if not p:
        return

    novo_nome = input(f"Novo Nome ({p['nome']}): ") or p["nome"]
    nova_idade = validar_int_positivo(
        f"Nova Idade ({p['idade']}): ", p["idade"]
    )
    novo_tel = input(f"Novo Tel ({p['telefone']}): ") or p["telefone"]
    novo_end = input(f"Novo Endereço ({p['endereco']}): ") or p["endereco"]

    with get_db_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            UPDATE pacientes SET nome=?, idade=?, telefone=?, endereco=? WHERE id=?
        """,
            (novo_nome, nova_idade, novo_tel, novo_end, p["id"]),
        )
        conn.commit()
    print("✅ Dados atualizados!")


def excluir_paciente():
    print("\n--- EXCLUIR PACIENTE ---")
    pacientes = listar_todos_pacientes()
    p = selecionar_paciente(pacientes)
    if not p:
        return

    confirmar = input(f"Confirmar exclusão de {p['nome']}? (S/N): ").upper()
    if confirmar == "S":
        with get_db_connection() as conn:
            conn.execute("DELETE FROM pacientes WHERE id=?", (p["id"],))
            conn.commit()
        print("🗑️ Paciente removido.")


def ver_estatisticas():
    pacientes = get_pacientes_para_exibir()
    if not pacientes:
        print("Sem dados para estatísticas.")
        return
    idades = [p["idade"] for p in pacientes]
    media = sum(idades) / len(pacientes)
    print(f"\nTotal de pacientes: {len(pacientes)}")
    print(f"Idade média: {media:.1f} anos")


def imprimir_cadastro():
    print("\n--- SALVAR CADASTRO (TXT) ---")
    pacientes = listar_todos_pacientes()
    p = selecionar_paciente(pacientes)
    if not p:
        return

    data_hora = datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    conteudo = f"""
======================================
FICHA DE CADASTRO DO PACIENTE
======================================
Nome: {p['nome']}
CPF: {p['cpf']}
Idade: {p['idade']} anos
Telefone: {p['telefone']}
Endereço: {p['endereco']}
======================================
Gerado em: {data_hora}
"""
    nome_arq = f"cadastro_{p['cpf']}.txt"
    with open(nome_arq, "w", encoding="utf-8") as f:
        f.write(conteudo)
    print(f"📄 Salvo com sucesso: {nome_arq}")


def menu_principal():
    # Inicializa o banco (cria a tabela 'pacientes' caso ela não exista)
    init_db()

    while True:
        opts = [
            "1. Cadastrar paciente",
            "2. Buscar paciente",
            "3. Listar todos",
            "4. Editar paciente",
            "5. Excluir paciente",
            "6. Ver estatísticas",
            "7. Imprimir TXT",
            "8. Sair",
        ]
        print_box("SISTEMA ONG VIDA+", opts)
        esc = input("Opção: ")
        if esc == "1":
            cadastrar_paciente()
        elif esc == "2":
            buscar_paciente()
        elif esc == "3":
            listar_todos_pacientes()
        elif esc == "4":
            editar_paciente()
        elif esc == "5":
            excluir_paciente()
        elif esc == "6":
            ver_estatisticas()
        elif esc == "7":
            imprimir_cadastro()
        elif esc == "8":
            print("\nEncerrando o sistema... Até logo!")
            break


if __name__ == "__main__":
    try:
        menu_principal()
    except KeyboardInterrupt:
        print("\n\n👋 Execução interrompida pelo usuário.")

Banco de dados 'clinica.db' verificado/inicializado.

╔════════════════════════════════════════════════════════════════════════════╗
║                             SISTEMA ONG VIDA+                              ║
╠════════════════════════════════════════════════════════════════════════════╣
║ 1. Cadastrar paciente                                                      ║
║ 2. Buscar paciente                                                         ║
║ 3. Listar todos                                                            ║
║ 4. Editar paciente                                                         ║
║ 5. Excluir paciente                                                        ║
║ 6. Ver estatísticas                                                        ║
║ 7. Imprimir TXT                                                            ║
║ 8. Sair                                                                    ║
╚════════════════════════════════════════════════════════════════════════════